# 6b. Inter-annotator agreement and adjudication

| | |
|---|---|
| **Stage** | 6b of 9 |
| **Purpose** | Compute agreement between annotator 1 (gold) and annotator 2 on the double-annotated sample, list the disagreements, adjudicate them in an interactive tool (also used for the S5 validation items), and summarise what changes in the training table. |
| **Inputs** | `outputs/supplementary/iaa_key_gold_DO_NOT_SHARE.csv, iaa_annotator2.csv`<br>`outputs/supplementary/iaa_key_topup_gold_DO_NOT_SHARE.csv, iaa_annotator2_topup.csv`<br>`scripts/aspect_taxonomy.py` |
| **Outputs** | `outputs/supplementary/iaa_disagreements.csv (adjudicated_label filled in by the tool)`<br>`outputs/supplementary/iaa_per_aspect.csv, iaa_long_table.csv, iaa_summary.json`<br>`outputs/supplementary/s5_adjudicated.csv (adjudication mode for the S5 items)` |
| **Runtime** | CPU only (about 20 seconds for the agreement statistics; adjudication is manual). |
| **Depends on** | Stage 6a (second annotator's files) and notebook 5 (the keys) |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

In [ ]:
# ===== S3b (standalone): inter-annotator agreement between annotator 1 (gold) and annotator 2 =====
# Needs: Google Drive with the GameSense folder, 4 csv files in outputs/supplementary, scripts/aspect_taxonomy.py. CPU only, ~20 s.
import os, sys, json
import numpy as np, pandas as pd
try:                                                    # mount Drive if running in Colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
OUT_DIR = f"{PROJECT_ROOT}/outputs/supplementary"
sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
from aspect_taxonomy import ASPECT_IDS
SENT_LABELS = ["negative", "neutral", "positive"]
L2I_ = {l: i for i, l in enumerate(SENT_LABELS)}

# (part name, annotator-1 key file, annotator-2 file). The key files hold the gold labels and were never shown to annotator 2.
PARTS = [("random60", "iaa_key_gold_DO_NOT_SHARE.csv", "iaa_annotator2.csv"),
         ("topup",    "iaa_key_topup_gold_DO_NOT_SHARE.csv", "iaa_annotator2_topup.csv")]
N_BOOT, MIN_POS = 2000, 5
OUTPUT_FILES = {"iaa_disagreements.csv", "iaa_disagreements_previous.csv", "iaa_per_aspect.csv", "iaa_long_table.csv", "iaa_summary.json"}
def out_path(name):                                     # this cell writes nothing except these files
    assert name in OUTPUT_FILES, name
    return f"{OUT_DIR}/{name}"
rng = np.random.default_rng(42)

# ---------- 0. Preflight ----------
need = [f for _, k, a in PARTS for f in (k, a)]
missing = [f for f in need if not os.path.exists(f"{OUT_DIR}/{f}")]
assert not missing, f"missing files in {OUT_DIR}: {missing}"

# ---------- 1. Long table: one row per (sentence, applicable aspect) ----------
rows, excluded = [], 0
for part, key_f, ann_f in PARTS:
    key = pd.read_csv(f"{OUT_DIR}/{key_f}")
    ann = pd.read_csv(f"{OUT_DIR}/{ann_f}")
    assert set(key["sentence_uid"]) == set(ann["sentence_uid"]), f"{part}: sentence sets differ between key and annotator file"
    if "_done" in ann.columns:                          # unconfirmed sentences would look like "nothing mentioned": drop them
        done = pd.to_numeric(ann["_done"], errors="coerce").fillna(0).astype(int) == 1
        print(f"{part}: {len(ann)} sentences, confirmed by annotator 2: {int(done.sum())}")
        if not done.all():
            print(f"  WARNING: {int((~done).sum())} sentences are not confirmed and are excluded from the statistics")
            excluded += int((~done).sum())
            ann = ann[done]
            key = key[key["sentence_uid"].isin(ann["sentence_uid"])]
    m = key.merge(ann[["sentence_uid", "applicable_aspects"] + [f"ann_{a}" for a in ASPECT_IDS if f"ann_{a}" in ann.columns]], on="sentence_uid")
    for _, r in m.iterrows():
        for a in [x.strip() for x in str(r["applicable_aspects"]).split(",") if x.strip()]:
            p1 = pd.to_numeric(r.get(f"gold_{a}"), errors="coerce") == 1
            s1 = str(r.get(f"gold_{a}_sentiment", "")).strip().lower() if p1 else ""
            s2 = str(r.get(f"ann_{a}", "")).strip().lower()
            s2 = s2 if s2 in SENT_LABELS else ""
            rows.append({"part": part, "sentence_uid": r["sentence_uid"], "game": r["game"], "aspect": a,
                         "p1": bool(p1), "p2": s2 != "", "s1": s1 if s1 in SENT_LABELS else "", "s2": s2,
                         "sentence_text": r["sentence_text"]})
L = pd.DataFrame(rows)
print(f"\ndecisions: {len(L)} (sentence x applicable aspect) | sentences: {L['sentence_uid'].nunique()}")

# ---------- 2. Statistics helpers ----------
def kappa_2x2(c):                                       # c = [n11, n10, n01, n00]
    n = c.sum(-1); po = (c[..., 0] + c[..., 3]) / n
    p1, p2 = (c[..., 0] + c[..., 1]) / n, (c[..., 0] + c[..., 2]) / n
    pe = p1 * p2 + (1 - p1) * (1 - p2)
    return np.where(pe < 1, (po - pe) / np.where(pe < 1, 1 - pe, 1), np.nan)
def ac1_2x2(c):                                         # Gwet's AC1: robust to low prevalence
    n = c.sum(-1); po = (c[..., 0] + c[..., 3]) / n
    pi = ((c[..., 0] + c[..., 1]) / n + (c[..., 0] + c[..., 2]) / n) / 2
    pe = 2 * pi * (1 - pi)
    return (po - pe) / np.where(pe < 1, 1 - pe, 1)
def pa_pos(c):                                          # positive agreement = 2*TP / (2*TP + FP + FN)
    d = 2 * c[..., 0] + c[..., 1] + c[..., 2]
    return np.where(d > 0, 2 * c[..., 0] / np.where(d > 0, d, 1), np.nan)
def counts(d):
    return np.array([(d.p1 & d.p2).sum(), (d.p1 & ~d.p2).sum(), (~d.p1 & d.p2).sum(), (~d.p1 & ~d.p2).sum()], float)
def kappa_table(T):                                     # k x k confusion table
    n = T.sum()
    if n == 0:
        return np.nan
    po = np.trace(T) / n; pe = (T.sum(0) * T.sum(1)).sum() / n ** 2
    return (po - pe) / (1 - pe) if pe < 1 else np.nan
def conf3(d):
    T = np.zeros((3, 3))
    np.add.at(T, (d["s1"].map(L2I_).values.astype(int), d["s2"].map(L2I_).values.astype(int)), 1)
    return T
def negrest(d):                                         # 2x2: negative vs not negative
    a, b = d.s1 == "negative", d.s2 == "negative"
    return np.array([[(a & b).sum(), (a & ~b).sum()], [(~a & b).sum(), (~a & ~b).sum()]], float)
def boot_ci(d, stat, key="sentence_uid"):               # cluster bootstrap over sentences
    groups = [g for _, g in d.groupby(key)]
    out = []
    for _ in range(N_BOOT):
        pick = rng.integers(0, len(groups), len(groups))
        out.append(stat(pd.concat([groups[i] for i in pick])))
    return np.nanpercentile(out, [2.5, 97.5])
ci = lambda lo_hi: f"[{lo_hi[0]:.2f}, {lo_hi[1]:.2f}]"
R = L[L["part"] == "random60"]                          # the unbiased sample: used for the overall numbers (A and B)

# ================= A. Aspect mentioned or not (overall; random 60 only) =================
c = counts(R)
print("\n=== A. Aspect mentioned or not (overall; random 60 sentences only) ===")
print(f"decisions={len(R)} | annotator1 marks {int(R.p1.sum())} ({100 * R.p1.mean():.1f}%) | annotator2 marks {int(R.p2.sum())} ({100 * R.p2.mean():.1f}%)")
print(f"both={int(c[0])}  only1={int(c[1])}  only2={int(c[2])}  neither={int(c[3])}")
res = {"decisions": len(R), "n11": int(c[0]), "only1": int(c[1]), "only2": int(c[2]), "n00": int(c[3]),
       "percent_agreement": round(float((c[0] + c[3]) / c.sum()), 3), "positive_agreement": round(float(pa_pos(c)), 3),
       "kappa": round(float(kappa_2x2(c)), 3), "gwet_ac1": round(float(ac1_2x2(c)), 3)}
res["kappa_ci95"] = [round(float(x), 3) for x in boot_ci(R, lambda d: float(kappa_2x2(counts(d))))]
res["positive_agreement_ci95"] = [round(float(x), 3) for x in boot_ci(R, lambda d: float(pa_pos(counts(d))))]
print(f"% agreement={res['percent_agreement']:.3f} | positive agreement (F1-like)={res['positive_agreement']:.2f} {ci(res['positive_agreement_ci95'])}"
      f" | Cohen's kappa={res['kappa']:.2f} {ci(res['kappa_ci95'])} | Gwet AC1={res['gwet_ac1']:.2f}")
print("(kappa is pulled down when the aspect is rare; positive agreement and AC1 are the prevalence-robust companions)")

# ================= B. Sentiment, when both annotators marked the aspect (random 60) =================
both_marked = R[R["p1"] & R["p2"]]
B = both_marked[(both_marked["s1"] != "") & (both_marked["s2"] != "")]
print("\n=== B. Sentiment toward the aspect, when both annotators marked it (random 60) ===")
if len(both_marked) != len(B):
    print(f"({len(both_marked) - len(B)} pairs excluded: annotator 1 marked the aspect without a sentiment label)")
if len(B) >= 5:
    T = conf3(B); k3, k2 = kappa_table(T), kappa_table(negrest(B))
    res.update({"sentiment_pairs": len(B), "sentiment_agreement": round(float(np.trace(T) / T.sum()), 3),
                "sentiment_kappa_3class": round(float(k3), 3), "sentiment_kappa_neg_vs_rest": round(float(k2), 3)})
    res["sentiment_kappa_3class_ci95"] = [round(float(x), 3) for x in boot_ci(B, lambda d: kappa_table(conf3(d)))]
    res["sentiment_kappa_negrest_ci95"] = [round(float(x), 3) for x in boot_ci(B, lambda d: kappa_table(negrest(d)))]
    print(f"pairs={len(B)} | agreement={res['sentiment_agreement']:.2f} | kappa 3-class={k3:.2f} {ci(res['sentiment_kappa_3class_ci95'])}"
          f" | kappa negative-vs-rest={k2:.2f} {ci(res['sentiment_kappa_negrest_ci95'])}")
    print("confusion (rows = annotator 1, cols = annotator 2; order negative / neutral / positive):")
    print(T.astype(int))
else:
    print(f"only {len(B)} pairs marked by both annotators: not enough for a kappa")

# ================= C. Per aspect (random 60 + top-up 20) =================
print("\n=== C. Per aspect (random 60 + top-up 20; the top-up is enriched for rare aspects, so use this table per aspect only) ===")
pa_rows = []
for a in ASPECT_IDS:
    d = L[L["aspect"] == a]
    if d.empty:
        continue
    cc = counts(d); bm = d[d.p1 & d.p2 & (d.s1 != "") & (d.s2 != "")]
    pa_rows.append({"aspect": a, "sentences": len(d), "A1_marks": int(d.p1.sum()), "A2_marks": int(d.p2.sum()), "both": int(cc[0]),
                    "positive_agreement": round(float(pa_pos(cc)), 2),
                    "kappa": round(float(kappa_2x2(cc)), 2) if min(d.p1.sum(), d.p2.sum()) >= MIN_POS else "n small",
                    "sentiment_agree": f"{int((bm.s1 == bm.s2).sum())}/{len(bm)}"})
pa = pd.DataFrame(pa_rows)
print(pa.to_string(index=False))
print(f"(kappa is shown only when both annotators marked the aspect at least {MIN_POS} times)")

# ================= D. Disagreement list (saved for adjudication; only the counts are printed) =================
D = L[(L.p1 != L.p2) | (L.p1 & L.p2 & (L.s1 != L.s2))].copy()
D["annotator1"] = np.where(D.p1, D.s1.replace("", "(marked, no sentiment)"), "-")
D["annotator2"] = np.where(D.p2, D.s2, "-")
D["type"] = np.select([D.p1 & ~D.p2, ~D.p1 & D.p2], ["only annotator 1 marked it", "only annotator 2 marked it"], "sentiment differs")
D["adjudicated_label"], D["note"] = "", ""             # <- to be filled with annotator 2: negative / neutral / positive / none
D = D[["part", "type", "game", "aspect", "annotator1", "annotator2", "adjudicated_label", "note", "sentence_uid", "sentence_text"]].sort_values(["type", "aspect"])
# Never lose adjudications: if the previous file already has filled adjudicated_label values, carry them over (matched on sentence_uid + aspect)
dpath = out_path("iaa_disagreements.csv")
if os.path.exists(dpath):
    prev = pd.read_csv(dpath)
    filled = prev[prev["adjudicated_label"].fillna("").astype(str).str.strip() != ""]
    if len(filled):
        prev.to_csv(out_path("iaa_disagreements_previous.csv"), index=False)          # safety copy of the file as it was
        kp = filled.drop_duplicates(["sentence_uid", "aspect"]).set_index(["sentence_uid", "aspect"])[["adjudicated_label", "note"]]
        D = D.set_index(["sentence_uid", "aspect"])
        common = D.index.intersection(kp.index)
        D.loc[common, "adjudicated_label"] = kp.loc[common, "adjudicated_label"].values
        D.loc[common, "note"] = kp.loc[common, "note"].fillna("").values
        D = D.reset_index()[["part", "type", "game", "aspect", "annotator1", "annotator2", "adjudicated_label", "note", "sentence_uid", "sentence_text"]]
        print(f"adjudications carried over: {len(common)} of {len(filled)} (the rest are no longer disagreements; see iaa_disagreements_previous.csv)")
D.to_csv(dpath, index=False)
pa.to_csv(out_path("iaa_per_aspect.csv"), index=False)
L.to_csv(out_path("iaa_long_table.csv"), index=False)
json.dump({**res, "per_aspect": pa.to_dict(orient="records"), "n_disagreements": int(len(D)), "excluded_unconfirmed_sentences": excluded},
          open(out_path("iaa_summary.json"), "w"), indent=2, default=str)
sp = f"{PROJECT_ROOT}/outputs/bert_summary_v2.json"      # also register the result in the project summary, if it exists
if os.path.exists(sp):
    s = json.load(open(sp)); s["iaa"] = {**res, "per_aspect": pa.to_dict(orient="records"), "n_disagreements": int(len(D))}
    json.dump(s, open(sp, "w"), indent=2, default=str)
print(f"\n=== D. {len(D)} disagreements saved to iaa_disagreements.csv ===")
print(D["type"].value_counts().to_string())
print("\nSaved: iaa_disagreements.csv, iaa_per_aspect.csv, iaa_long_table.csv, iaa_summary.json")

In [ ]:
# ===== Adjudication tool (CPU only): decide the final label for each disagreement; every click is saved to the csv =====
# MODE "iaa": outputs/supplementary/iaa_disagreements.csv (column adjudicated_label: negative / neutral / positive / none)
# MODE "s5" : outputs/supplementary/s5_adjudicated.csv    (column final_label:        negative / neutral / positive / na)
import os, sys, html
import numpy as np, pandas as pd
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

MODE = "s5"                                   # <- "iaa" or "s5"
PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
SUPP = f"{PROJECT_ROOT}/outputs/supplementary"
CFG = {"iaa": dict(file="iaa_disagreements.csv", col="adjudicated_label", none="none"),
       "s5":  dict(file="s5_adjudicated.csv",    col="final_label",       none="na")}[MODE]
PATH = f"{SUPP}/{CFG['file']}"
LABEL_COL, NONE_LABEL = CFG["col"], CFG["none"]

try:                                           # aspect definitions from the official taxonomy (same text both annotators saw)
    sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
    from aspect_taxonomy import ASPECT_TAXONOMY
except Exception:
    ASPECT_TAXONOMY = {}

def adjudicate():
    df = pd.read_csv(PATH, dtype=object).fillna("")
    for c in [LABEL_COL, "note"]:
        if c not in df.columns:
            df[c] = ""
    n = len(df)
    # default order: by aspect, so similar cases are decided together
    order = df.sort_values(["aspect"] + (["type"] if "type" in df.columns else []), kind="stable").index.tolist()
    state = {"pos": 0}
    head, body, info = widgets.HTML(), widgets.HTML(), widgets.HTML()
    note = widgets.Text(placeholder="optional note (the rule you applied)", layout=widgets.Layout(width="620px"))
    rules = widgets.HTML(
        "<div style='font-size:12px;color:#999;line-height:1.6;margin-top:8px'>"
        "<b>Rules we agreed:</b> judge the attitude toward THIS aspect &bull; "
        f"<b>{NONE_LABEL}</b> = the aspect is not really discussed (also pure emotion / feature wishes without a complaint / truncated fragments) &bull; "
        "negating a problem ('no ads', 'no cheaters') is <b>positive</b> &bull; problem resolved / thanking support is <b>positive</b> &bull; "
        "performance praise counts only if attributable to the game itself (smoothness may depend on the player's device or network) &bull; "
        "neutral = factual or mixed, no evaluative stance.</div>")

    def done_count():
        return int((df[LABEL_COL].astype(str).str.strip() != "").sum())

    def show():
        i = order[state["pos"]] if state["pos"] < n else None
        head.value = (f"<b>Item {min(state['pos'] + 1, n)}/{n}</b> &nbsp; <span style='color:#999'>adjudicated so far: {done_count()}/{n}</span>")
        if i is None:
            body.value = f"<h3>End of list. Adjudicated: {done_count()}/{n}. Use 'Back' to review or 'Skip' items you left open.</h3>"
            info.value = ""
            return
        r = df.loc[i]
        meta = ASPECT_TAXONOMY.get(r["aspect"], {})
        a1, a2 = r.get("annotator1", ""), r.get("annotator2", "")
        cur = r[LABEL_COL]
        body.value = (f"<p style='font-size:19px'>{html.escape(str(r['sentence_text']))}</p>"
                      f"<p style='color:#8fd0ff'>Aspect: <b>{html.escape(str(meta.get('label', r['aspect'])))}</b>"
                      f"{' &middot; ' + html.escape(str(r['game'])) if 'game' in df.columns else ''}"
                      f"{' &middot; ' + html.escape(str(r['type'])) if 'type' in df.columns else ''}<br>"
                      f"<span style='font-size:12px;color:#aaa'>{html.escape(str(meta.get('description', '')))}</span></p>"
                      f"<p>Annotator 1: <b>{html.escape(str(a1))}</b> &nbsp;|&nbsp; Annotator 2: <b>{html.escape(str(a2))}</b>"
                      f"{' &nbsp;|&nbsp; <span style=\"color:#7fd18b\">current decision: <b>' + html.escape(cur) + '</b></span>' if cur else ''}</p>")
        note.value = r["note"]

    def save():
        df.to_csv(PATH, index=False)

    def decide(label):
        if state["pos"] >= n:
            return
        i = order[state["pos"]]
        df.at[i, LABEL_COL] = label
        df.at[i, "note"] = note.value
        save()
        state["pos"] += 1
        show()

    def skip(_):
        if state["pos"] < n:
            state["pos"] += 1
            show()

    def back(_):
        if state["pos"] > 0:
            state["pos"] -= 1
            show()

    def clear(_):                                # remove the decision of the current item
        if state["pos"] < n:
            i = order[state["pos"]]
            df.at[i, LABEL_COL], df.at[i, "note"] = "", ""
            save()
            show()

    buttons = []
    for lab, style, text in [("negative", "danger", "negative"), ("neutral", "", "neutral"), ("positive", "success", "positive"),
                             (NONE_LABEL, "warning", f"{NONE_LABEL} (aspect not discussed)")]:
        b = widgets.Button(description=text, button_style=style, layout=widgets.Layout(width="200px"))
        b.on_click(lambda _, v=lab: decide(v))
        buttons.append(b)
    nav = []
    for text, fn in [("Back", back), ("Skip", skip), ("Clear decision", clear)]:
        b = widgets.Button(description=text, layout=widgets.Layout(width="130px"))
        b.on_click(fn)
        nav.append(b)
    show()
    display(widgets.VBox([head, body, widgets.HBox(buttons), note, widgets.HBox(nav), rules]))
    return df, state, buttons, nav, note

_ = adjudicate()

In [ ]:
# ===== IAA adjudication summary (CPU only, read-only): how often was each annotator right, and what will change in the training table? =====
import os, sys
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass
PROJECT_ROOT = "/content/drive/MyDrive/GameSense"
SUPP = f"{PROJECT_ROOT}/outputs/supplementary"
SENT = ["negative", "neutral", "positive"]
VALID = SENT + ["none"]

L = pd.read_csv(f"{SUPP}/iaa_long_table.csv", dtype=object).fillna("")
D = pd.read_csv(f"{SUPP}/iaa_disagreements.csv", dtype=object).fillna("")
for c in ["p1", "p2"]:
    L[c] = L[c].astype(str).str.lower().eq("true")
D["adj"] = D["adjudicated_label"].astype(str).str.strip().str.lower()

# ---------- 0. Validity of the adjudication file ----------
bad = D[~D["adj"].isin(VALID)]
print(f"disagreements: {len(D)} | filled: {int(D['adj'].ne('').sum())} | blank: {int(D['adj'].eq('').sum())} | invalid labels: {len(bad)}")
if len(bad):
    print("rows with a blank or invalid adjudicated_label (allowed: negative / neutral / positive / none):")
    print(bad[["type", "aspect", "adjudicated_label", "sentence_text"]].to_string(index=False))
adj = D.drop_duplicates(["sentence_uid", "aspect"]).set_index(["sentence_uid", "aspect"])["adj"]

# ---------- 1. Final label of every (sentence, aspect) decision ----------
def final_label(r):
    if r["p1"] and r["p2"] and r["s1"] == r["s2"] and r["s1"] != "":
        return r["s1"]                                               # both marked, same sentiment: nothing to decide
    if (r["sentence_uid"], r["aspect"]) in adj.index:
        return adj[(r["sentence_uid"], r["aspect"])] or "?"          # adjudicated ('?' = still blank)
    return "none"                                                    # neither marked it
L["final"] = L.apply(final_label, axis=1)
L["final_marked"] = ~L["final"].isin(["none", "?"])
print("decisions still unresolved ('?'):", int((L["final"] == "?").sum()))

def quality(d, who):
    marks = d[who]
    tp = (marks & d["final_marked"]).sum()
    prec = tp / marks.sum() if marks.sum() else np.nan
    rec = tp / d["final_marked"].sum() if d["final_marked"].sum() else np.nan
    return int(marks.sum()), int(tp), round(float(prec), 2), round(float(rec), 2)

# ---------- 2. Headline numbers on the random 60 (unbiased sample) ----------
R = L[L["part"] == "random60"]
print("\n=== How good are the labels against the adjudicated result? (random 60 sentences) ===")
print(f"adjudicated mentions (final positives): {int(R['final_marked'].sum())}")
for who, name in [("p1", "Annotator 1 (gold)"), ("p2", "Annotator 2")]:
    n, tp, pr, rc = quality(R, who)
    print(f"{name:20s} marks={n:3d}  confirmed={tp:3d}  precision={pr:.2f}  recall={rc:.2f}")
g = R[R["p1"] & R["final_marked"]]
print(f"gold sentiment equals the adjudicated sentiment in {int((g['s1'] == g['final']).sum())}/{len(g)} confirmed mentions "
      f"({100 * (g['s1'] == g['final']).mean():.0f}%)")

# ---------- 3. Per aspect (all 80 sentences; the top-up is enriched, so reference only) ----------
rows = []
for a, d in L.groupby("aspect"):
    n1, tp1, p1_, r1_ = quality(d, "p1"); n2, tp2, p2_, r2_ = quality(d, "p2")
    rows.append({"aspect": a, "final_mentions": int(d["final_marked"].sum()), "gold_marks": n1, "gold_precision": p1_, "gold_recall": r1_,
                 "A2_marks": n2, "A2_precision": p2_, "A2_recall": r2_})
pa = pd.DataFrame(rows).sort_values("final_mentions", ascending=False)
pd.set_option("display.width", 250)
print("\n=== Per aspect (all 80 sentences) ===")
print(pa.to_string(index=False))

# ---------- 4. What the v3 training table will change (same rules as Cell 3 of the v3 notebook) ----------
dropped = L[L["p1"] & (L["final"] == "none")]
relab = L[L["p1"] & L["final_marked"] & (L["s1"] != L["final"])]
added = L[~L["p1"] & L["final_marked"]]
kept = L[L["p1"] & L["final_marked"] & (L["s1"] == L["final"])]
print("\n=== Effect on the gold sentiment pairs used for training (all 80 sentences) ===")
print(f"unchanged: {len(kept)} | relabeled: {len(relab)} | dropped (aspect not discussed): {len(dropped)} | added (marked only by annotator 2): {len(added)}")
if len(added):
    print("added by aspect:", added["aspect"].value_counts().to_dict())
if len(dropped):
    print("dropped by aspect:", dropped["aspect"].value_counts().to_dict())
if len(relab):
    print("relabeled by aspect:", relab["aspect"].value_counts().to_dict())